# Notebook 8 -- Three-Angle Evaluation on AQUA-RAT
## SLM-to-SLM Guided Reasoning Pipeline

**Why AQUA-RAT?**

AQUA-RAT (Algebraic Question Answering with Rationales) is a GRE/GMAT-level
algebra benchmark in **multiple-choice format**. It is significantly harder than
SVAMP or GSM8K -- problems require ratio reasoning, percentages, algebra, and
multi-step inference. Every question has 5 options (A to E).

| Property | SVAMP | AQUA-RAT |
|---|---|---|
| HuggingFace ID | `ChilleD/SVAMP` | `deepmind/aqua_rat` |
| Size (test) | 1,000 | 254 |
| Format | Open numeric | Multiple choice A-E |
| Difficulty | Grade school | GRE / GMAT algebra |
| Random baseline | N/A | 20% (1 in 5) |

**Key differences from SVAMP notebook**
- Answer is a letter (A/B/C/D/E), not a number
- Options are included in the question fed to the model
- Guide identifies the correct reasoning approach and which option to eliminate
- Extractor targets letter patterns, not numeric patterns
- Baseline chance level is 20% -- any result above 20% is non-trivial
- If guided pipeline scores well above baseline chance, the finding is strong

**Pipeline (identical structure)**
```
Question + Options --> Fine-tuned Qwen 3B (Guide) --> Plan --> Qwen 1.5B (Solver) x5 --> Vote --> Letter Answer
Baseline: Question + Options -----------------------------------------> Qwen 1.5B x5 --> Vote
```


In [1]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")


Done.


In [2]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print("HuggingFace login done")


HuggingFace login done


In [3]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/aqua_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")


PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB
Output  : /kaggle/working/aqua_eval


In [4]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "deepmind/aqua_rat",
    "dataset_split"       : "test",     # 254 test questions
    "max_eval_samples"    : 50,        # use all 254 for full run
    "random_seed"         : 42,         # FIXED -- same seed for BOTH conditions

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 800,        # slightly more -- algebra needs more steps

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


Config ready:
  guide_base              : Qwen/Qwen2.5-3B-Instruct
  response_model          : Qwen/Qwen2.5-1.5B-Instruct
  dataset_name            : deepmind/aqua_rat
  dataset_split           : test
  max_eval_samples        : 50
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.7
  guide_temperature       : 0.1
  refiner_temperature     : 0.3
  max_new_tokens          : 800
  guide_params_B          : 3.0
  solver_params_B         : 1.5
  results_file            : /kaggle/working/aqua_eval/results.jsonl
  report_file             : /kaggle/working/aqua_eval/eval_report.json
  angle1_file             : /kaggle/working/aqua_eval/angle1_compute_efficiency.json
  angle2_file             : /kaggle/working/aqua_eval/angle2_vote_consistency.json
  angle3_file             : /kaggle/working/aqua_eval/angle3_confidence_calibration.json
  checkpoint_file         : /kaggle/working/aqua_eval/checkpoint.json
  save_every              : 25


In [5]:
# CELL 5 -- Load AQUA-RAT dataset
# Fields: question, options (list of strings like "A)12", "B)15"...),
#         rationale, correct (single letter "A" to "E")
#
# We format the question as:
#   "<question text>\nOptions:\nA) ...\nB) ...\n..."
# This gives the model all the information it needs to pick a letter.

print("Loading AQUA-RAT from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits   : {list(raw_ds.keys())}")
print(f"Features : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Test size: {len(raw_ds[CONFIG['dataset_split']])}")

ex = raw_ds[CONFIG["dataset_split"]][0]
print(f"\nExample record:")
for k, v in ex.items():
    print(f"  {k}: {v}")


def format_options(options_list):
    """
    AQUA options come as ['A)12', 'B)25', ...] or ['A) 12', 'B) 25', ...].
    Normalise to clean 'A) 12' format and join as newline-separated string.
    """
    formatted = []
    for opt in options_list:
        opt = opt.strip()
        # Ensure space after letter: 'A)12' -> 'A) 12'
        opt = re.sub(r'^([A-E])\)', r'\1) ', opt)
        formatted.append(opt)
    return "\n".join(formatted)


def normalise_aqua(item):
    """Convert AQUA record to {question, answer} used by the pipeline."""
    options_str = format_options(item["options"])
    q = item["question"].strip() + "\n\nOptions:\n" + options_str
    # GT answer is a single uppercase letter: 'A', 'B', 'C', 'D', or 'E'
    ans = item["correct"].strip().upper()
    return {"question": q, "answer": ans, "rationale": item.get("rationale", "")}


all_data = [normalise_aqua(x) for x in raw_ds[CONFIG["dataset_split"]]]

# CRITICAL: fix seed ONCE here, before any sampling
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

# Distribution of correct answers -- should be roughly uniform A-E
ans_dist = Counter(d["answer"] for d in test_data)
print(f"\nAnswer distribution (should be ~uniform):")
for letter in "ABCDE":
    print(f"  {letter}: {ans_dist.get(letter, 0)}")

print(f"\nFirst Q  : {test_data[0]['question'][:120]}...")
print(f"First A  : {test_data[0]['answer']}")
print("AQUA-RAT loaded")


Loading AQUA-RAT from HuggingFace...


README.md: 0.00B [00:00, ?B/s]

raw/train-00000-of-00001.parquet:   0%|          | 0.00/25.4M [00:00<?, ?B/s]

raw/test-00000-of-00001.parquet:   0%|          | 0.00/74.0k [00:00<?, ?B/s]

raw/validation-00000-of-00001.parquet:   0%|          | 0.00/76.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/97467 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/254 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/254 [00:00<?, ? examples/s]

Splits   : ['train', 'test', 'validation']
Features : ['question', 'options', 'rationale', 'correct']
Test size: 254

Example record:
  question: A car is being driven, in a straight line and at a uniform speed, towards the base of a vertical tower. The top of the tower is observed from the car and, in the process, it takes 10 minutes for the angle of elevation to change from 45° to 60°. After how much more time will this car reach the base of the tower?
  options: ['A)5(√3 + 1)', 'B)6(√3 + √2)', 'C)7(√3 – 1)', 'D)8(√3 – 2)', 'E)None of these']
  rationale: Explanation :
Let the height of the building be h. Initially, he was at an angle of 450. tan 45 = h/distance between car and tower. h = distance between car and tower (since tan 45 = 1).
Now, after 10 minutes, it travelled a certain distance, and angle changed to 600.
tan 60 = h/x x = h/√3
So, in 10 minutes, it has travelled a distance of h – x = h - h/√3.
10 minutes = h *( 1 – 1√3)
h can be travelled in 10 / (1 – 1√3).
To travel a 

In [6]:
# CELL 6 -- Answer extraction for multiple-choice (A-E)
# AQUA-RAT answers are single letters A, B, C, D, or E.
# We need to extract which letter the model chose from free-form text.

VALID_LETTERS = set("ABCDE")


def extract_gt_answer(answer_str):
    """GT is already a clean letter -- just uppercase and validate."""
    s = str(answer_str).strip().upper()
    return s if s in VALID_LETTERS else ""


def extract_pred_answer(text):
    """
    Extract the chosen option letter (A-E) from model output.
    Priority order -- most explicit formats first.
    Never fires on option listings like 'Option A: 4 - Not divisible'.
    """
    text = text.strip()

    # 1. Conclusive answer phrases only (NOT "option" / "choice" -- those fire on listings)
    #    Matches: "The answer is A", "Answer: B", "correct answer is C", "answer is D"
    m = re.search(
        r"(?:the answer is|answer is|answer:|the correct answer is|correct answer is)"
        r"[\s:]*([A-E])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 2. "option/choice X is correct/matches/is the answer"
    #    ONLY fires when explicitly confirmed -- NOT on "Option A: 4 - not divisible"
    m = re.search(
        r"(?:option|choice)\s+([A-E])\s+(?:is correct|is the answer|matches|is right)",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 3. #### A  -- standard termination marker
    m = re.search(r"####\s*([A-E])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 4. Parenthesised at end of text: (A), (B) ...
    #    Must be at the very end to avoid matching "(A) 120" mid-sentence
    m = re.search(r"\(([A-E])\)\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 5. Bold: **A**, **A)**
    m = re.search(r"\*\*([A-E])\)?\*\*", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 6. "therefore/thus/hence/so A" conclusion phrases
    m = re.search(
        r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?([A-E])\b",
        text, re.IGNORECASE
    )
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 7. "Target value: 14 -- matches D" (new guide format)
    m = re.search(r"matches\s+([A-E])\b", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # 8. Standalone letter at the very end of output ONLY
    #    Guard: the char before must NOT be ) or a digit (avoids "A)120" or "Step 2A")
    m = re.search(r"(?<![)\d])\b([A-E])\s*$", text, re.IGNORECASE)
    if m and m.group(1).upper() in VALID_LETTERS:
        return m.group(1).upper()

    # No valid letter found
    return ""

# --- Self-test ---
_tests = [
    ("The answer is A",          "A"),
    ("Answer: C",                "C"),
    ("#### B",                   "B"),
    ("The correct answer is D",  "D"),
    ("Option E",                 "E"),
    ("(B)",                      "B"),
    ("**C)**",                   "C"),
    ("Therefore, D",             "D"),
    ("Thus B is correct",        "B"),
    ("Some unrelated text",      ""),
    ("The answer is 42",         ""),   # number should not match
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:40]}' -> '{got}' (expected '{exp}')")
print("\nAll extractor tests passed" if ok else "\nEXTRACTOR HAS FAILURES -- fix before running eval")


  OK  'The answer is A' -> 'A' (expected 'A')
  OK  'Answer: C' -> 'C' (expected 'C')
  OK  '#### B' -> 'B' (expected 'B')
  OK  'The correct answer is D' -> 'D' (expected 'D')
  OK  'Option E' -> 'E' (expected 'E')
  OK  '(B)' -> 'B' (expected 'B')
  OK  '**C)**' -> 'C' (expected 'C')
  OK  'Therefore, D' -> 'D' (expected 'D')
  OK  'Thus B is correct' -> 'B' (expected 'B')
  OK  'Some unrelated text' -> '' (expected '')
  OK  'The answer is 42' -> '' (expected '')

All extractor tests passed


In [7]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token

guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")

guide_model.eval()
print(f"Guide VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading guide base: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Found adapter: /kaggle/input/datasets/sufiantabdullah/final-adapter
LoRA adapter loaded -- fine-tuned guide active
Guide VRAM: 6.29 GB


In [8]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models): {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                : {headroom:.1f} GB")
print("Memory OK" if headroom >= 2 else "WARNING: Tight -- reduce n_votes to 3 if OOM")


Loading solver: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total VRAM (both models): 9.38 GB / 17.1 GB
Headroom                : 7.7 GB
Memory OK


In [9]:
# CELL 9 -- Prompts and generation functions

GUIDE_SYSTEM = (
    """
    You are a math reasoning planner for multiple-choice quantitative problems (AQUA-RAT).

    Your job is to produce a clear numbered reasoning plan that a solver can follow.
    
    Rules:
    
    1. Do NOT solve the problem.
    2. Do NOT compute any numbers.
    3. Do NOT choose an answer option.
    4. Only describe what calculations or logical steps must be performed.
    
    Plan structure:
    
    1. State the goal of the problem in one short sentence.
    2. List the key quantities or variables given in the problem.
    3. Describe how the quantities relate to each other.
    4. Describe the intermediate values that must be computed.
    5. Describe the formula or method that will produce the final result.
    6. State that the final numeric result should be compared with the answer choices.
    
    Formatting rules:
    
    * Output numbered steps only.
    * No markdown.
    * No bullet points.
    * No symbols like *, #, or LaTeX.
    * No explanations outside the numbered list.
    
    Example style:
    
    1. Goal: Determine the final distance from the starting point.
    2. Identify movements in each direction and group them by axis.
    3. Determine the net north-south displacement.
    4. Determine the net east-west displacement.
    5. Use the distance formula to compute the straight-line distance from the origin.
    6. Compare the computed value with the answer choices.

    """
)

SOLVE_SYSTEM = (
    "You are a multiple-choice math solver.\n"
    "Solve the problem yourself from scratch.\n"
    "follow given plan strictly\n"
    "Your own arithmetic is authoritative.\n"
    "No markdown. Show every calculation.\n"
    "Your absolute last line must be exactly: answer is [letter]"
)

BASELINE_SYSTEM = (
    "You are a precise multiple-choice math solver.\n"
    "Read the problem and all options carefully.\n"
    "Solve step by step showing every calculation.\n"
    "Your absolute last line must be exactly: The answer is [letter]"
)

REFINER_SYSTEM = (
    "You are a careful multiple-choice math solver.\n"
    "Previous attempts gave different answers. Ignore them completely.\n"
    "Re-solve from scratch using only the problem and options given.\n"
    "Show every step. Your last line must be: The answer is [letter]"
)



def get_markdown_banned_ids(tok):
    patterns = ["**", "###", "##", "* ", "- ", "• ",
                "\\(", "\\)", "\\[", "\\]",   # ← add these
                "\\frac", "\\times", "\\text"] # ← add these
    banned = []
    for p in patterns:
        ids = tok.encode(p, add_special_tokens=False)
        if ids:
            banned.append(ids)
    return banned

BANNED_IDS = get_markdown_banned_ids(resp_tok)


def run_qwen(mdl, tok, messages, max_tokens, temperature):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1280)
    device = next(mdl.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.15,
            bad_words_ids      = BANNED_IDS,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_plan(question):
    return run_qwen(
        guide_model, guide_tok,
        [{"role": "system", "content": GUIDE_SYSTEM},
         {"role": "user",   "content": f"Problem:\n{question}"}],
        max_tokens  = 350,
        temperature = CONFIG["guide_temperature"],
    )


def generate_guided(question, plan):
    content = (
        f"Problem:\n{question}\n\n"
        f"Reasoning plan:\n{plan}\n\n"
        f"Solve from scratch. Your calculation is authoritative:"
    )
    for _ in range(2):
        raw = run_qwen(
            resp_model, resp_tok,
            [{"role": "system", "content": SOLVE_SYSTEM},
             {"role": "user",   "content": content}],
            max_tokens  = CONFIG["max_new_tokens"],
            temperature = CONFIG["vote_temperature"],
        )
        if extract_pred_answer(raw):
            return raw
    return raw


def generate_baseline(question):
    for _ in range(2):
        raw = run_qwen(
            resp_model, resp_tok,
            [{"role": "system", "content": BASELINE_SYSTEM},
             {"role": "user",   "content": f"Problem:\n{question}"}],
            max_tokens  = CONFIG["max_new_tokens"],
            temperature = CONFIG["vote_temperature"],
        )
        if extract_pred_answer(raw):
            return raw
    return raw


def generate_refiner(question, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem:\n{question}\n\n"
        f"Previous attempts disagreed: {cands}\n"
        "Ignore all previous attempts. Solve from scratch:"
    )
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")

Generation functions ready


In [10]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Filters empty/invalid letter responses before counting.

    Returns dict with all metrics needed for the three angles.
    """
    # Filter to valid letters only
    valid = [a for a in answers if a in VALID_LETTERS]
    if not valid:
        valid = answers  # fallback -- keep all if extraction completely failed

    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(valid)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / max(total, 1)
    is_majority      = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: refiner runs WITHOUT plan
        ref_raw  = generate_refiner(question, list(valid))
        ref_ans  = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_v      = valid + ([ref_ans] if ref_ans in VALID_LETTERS else [])
        new_counts = Counter(all_v)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_v), 4)
        total      = len(all_v)
        correct_votes    = Counter(all_v).get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / max(total, 1)
        wasted           = total - new_top_c
        vote_counts      = new_counts

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")
print("  majority         -> clear letter winner")
print("  refiner_tiebreak -> tie broken by refiner (no plan)")
print("  coin_flip        -> still tied after refiner")


Voting logic ready
  majority         -> clear letter winner
  refiner_tiebreak -> tie broken by refiner (no plan)
  coin_flip        -> still tied after refiner


In [11]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (AQUA-RAT)")
print("=" * 65)

item = test_data[20]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question:\n{q}")
print(f"\nGT Answer: {gt}")

# Guided
print("\n\n\n\n🪼[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

SINGLE QUESTION TEST  (AQUA-RAT)
Question:
Train A leaves a station every 16 minutes and Train B leaves every 17 minutes. If both trains just left the station simultaneously, how long until they do so again?

Options:
A) 272 minutes
B) 304 minutes
C) 190 minutes
D) 70 minutes
E) 35 minutes

GT Answer: A




🪼[1] Guide generating plan...
Plan:
1. Goal: Find when both Trains A and B leave the station at the same time next.
2. Key quantities: Time interval for Train A = 16 min; Time interval for Train B = 17 min.
3. Relationship between intervals shows their departure times coincide periodically based on least common multiple (LCM).
4. Compute LCM using prime factorization if needed but here it's straightforward as no common factors beyond themselves exist directly observable through inspection.
5. Formula/method used would involve finding multiples until matching occurs which isn't explicitly required computation-wise since we know there’s no overlap without further calculation due simpl

In [12]:
print(f"🪬[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  *********Vote {i+1}: '{pred}'***********  |  \n\nraw: {raw}\n\n")

g = vote_and_decide(guided_votes, q, gt)
print(f"\n  Result    : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy  : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")
print(f"  Vote counts: {g['vote_counts']}")

🪬[2] Guided votes (5x)...
  *********Vote 1: ''***********  |  

raw: To find out when both trains will depart the station at the same time next:

- Train A departs every 16 minutes.
- Train B departs every 17 minutes.

We need to determine the smallest number of minutes after this event that both trains will arrive back at the station in perfect synchronization with each other.

Since these train schedules differ by only one minute (17 - 16 = 1), every subsequent round trip where both trains travel the same distance will cause them to meet up at the station again. 

The first time this happens is simply 16 + 17 = **33 minutes**, because that covers the gap between the two schedules plus another full cycle of either train leaving.

Therefore, the correct option is not listed among those choices, indicating an error or misunderstanding in reading comprehension regarding the logical connection implied by our reasoning step. The mathematical solution leads us to conclude they won’t match 

In [13]:
# Baseline
print("🧢[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  *********Vote {i+1}: '{pred}'*********  |  \nraw: {raw}\n\n")

b = vote_and_decide(base_votes, q, gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print(f"  Vote counts: {b['vote_counts']}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


🧢[3] Baseline votes (no plan)...
  *********Vote 1: 'A'*********  |  
raw: To determine when both trains will leave the station simultaneously again, we need to find the least common multiple (LCM) of their intervals.

The interval for Train A is 16 minutes.
The interval for Train B is 17 minutes.

Let's calculate the LCM:

- First, factor each number into its prime factors:
    - \( \text{Prime Factors of } 16 = 2^4 \)
    - \( \text{Prime Factor of } 17 = 17^1 \)

Since 17 is already in its simplest form as an integer, it remains 17.

- For two numbers with different bases, multiply together the highest power of each base that appears:
    - LCM(16, 17) = \(2^4 \times 17^1\)

Now let’s compute this product:
   - \( 2^4 = 16 \)
   - \( 16 \times 17 = 272 \)

Therefore, the Least Common Multiple (LCM) of 16 and 17 is **272 minutes**, which means the trains will leave the station simultaneously again after 272 minutes.

Final Answer:  
The answer is A) 272 minutes


  *********Vote 2: '

In [14]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided  (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)

print(f"Dual evaluation: {len(test_data)} AQUA-RAT questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print(f"Random baseline (chance): 20.0% (1 in 5 options)")
print("-" * 65)

all_results  = []
base_results = []
start_idx    = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="AQUA-RAT Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -----------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "correct_votes"    : g_dec["correct_votes"],
            "total_votes"      : g_dec["total_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
            "vote_counts"      : g_dec["vote_counts"],
            "plan"             : plan,
        })
    except RuntimeError as e:
        all_results.append({
            "mode": "guided", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # ---- BASELINE ---------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "correct_votes"    : b_dec["correct_votes"],
            "total_votes"      : b_dec["total_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : None,
            "vote_counts"      : b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Guided: {g_acc:.1f}%  Baseline: {b_acc:.1f}%  (random=20%)  ({mins:.1f} min)")

with open(CONFIG["results_file"], "w") as f:
    for r in all_results + base_results:
        f.write(json.dumps(r) + "\n")

g_c = sum(r["correct"] for r in all_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nEvaluation complete.")
print(f"  Guided   : {g_c}/{len(all_results)} = {g_c/len(all_results)*100:.1f}%")
print(f"  Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  Random   : 20.0% (chance)")
print(f"  Delta    : +{(g_c/len(all_results) - b_c/len(base_results))*100:.1f} percentage points") 


Dual evaluation: 50 AQUA-RAT questions
Each question: 5 guided votes + 5 baseline votes
Random baseline (chance): 20.0% (1 in 5 options)
-----------------------------------------------------------------
Starting fresh


AQUA-RAT Eval:   0%|          | 0/50 [00:00<?, ?it/s]

  [ 25] Guided: 56.0%  Baseline: 40.0%  (random=20%)  (103.0 min)
  [ 50] Guided: 52.0%  Baseline: 40.0%  (random=20%)  (201.7 min)

Evaluation complete.
  Guided   : 26/50 = 52.0%
  Baseline : 20/50 = 40.0%
  Random   : 20.0% (chance)
  Delta    : +12.0 percentage points


In [15]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
G, S, N = CONFIG["guide_params_B"], CONFIG["solver_params_B"], CONFIG["n_votes"]

guided_compute   = (G * 1) + (S * N)
baseline_compute = S * N
upper_compute    = G * N

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
random_chance = 20.0   # 1 in 5 options

g_eff = g_acc / guided_compute
b_eff = b_acc / baseline_compute
savings_pct = (1 - guided_compute / upper_compute) * 100

g_wasted = sum(r["wasted_votes"] for r in all_results)
b_wasted = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

ref_triggered = sum(r["refiner_used"] for r in all_results)
ref_correct   = sum(1 for r in all_results if r["refiner_used"] and r.get("refiner_correct"))

strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats: strategy_stats[s] = {"n":0,"correct":0}
    strategy_stats[s]["n"] += 1
    if r["correct"]: strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (AQUA-RAT)")
print("=" * 65)
print(f"  Random chance baseline: {random_chance}% (5 options, A-E)")
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}")
print(f"  {'Random chance':<32} | {'--':>10} | {random_chance:>8.1f}%")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}%")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}%")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9}")
print(f"\n  Guided vs baseline gain : +{g_acc - b_acc:.1f} pts")
print(f"  Guided vs random chance : +{g_acc - random_chance:.1f} pts above chance")
print(f"  Compute savings vs upper: {savings_pct:.0f}% cheaper")
print(f"  Wasted votes saved      : {b_wasted - g_wasted}")
if ref_triggered:
    print(f"  Refiner: {ref_triggered} triggered, {ref_correct} correct ({ref_correct/ref_triggered*100:.1f}%)")
print(f"\n  Strategy breakdown:")
for s, v in sorted(strategy_stats.items(), key=lambda x: -x[1]["n"]):
    acc_s = v["correct"]/v["n"]*100 if v["n"] else 0
    print(f"    {s:<22}: {v['n']:>4} questions  {acc_s:>6.1f}% accuracy")

angle1 = {
    "dataset": "AQUA-RAT", "n_questions": len(all_results),
    "random_chance_pct": random_chance,
    "guided_compute_B": guided_compute, "baseline_compute_B": baseline_compute,
    "upper_compute_B": upper_compute, "guided_accuracy": round(g_acc,2),
    "baseline_accuracy": round(b_acc,2), "accuracy_gain": round(g_acc-b_acc,2),
    "guided_above_chance": round(g_acc-random_chance,2),
    "baseline_above_chance": round(b_acc-random_chance,2),
    "compute_savings_pct": round(savings_pct,1),
    "guided_wasted_votes": g_wasted, "baseline_wasted_votes": b_wasted,
    "wasted_votes_saved": b_wasted-g_wasted,
    "refiner_triggered": ref_triggered, "refiner_correct": ref_correct,
    "strategy_breakdown": strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")


ANGLE 1 -- COMPUTE EFFICIENCY  (AQUA-RAT)
  Random chance baseline: 20.0% (5 options, A-E)

  Setup                            |    Compute |  Accuracy
  ---------------------------------+------------+----------
  Random chance                    |         -- |     20.0%
  Baseline (1.5B x 5)              |      7.5B  |     40.0%
  Guided  (3B x1 + 1.5B x5)        |     10.5B  |     52.0%
  Upper   (3B x 5)                 |     15.0B  | (ceiling)

  Guided vs baseline gain : +12.0 pts
  Guided vs random chance : +32.0 pts above chance
  Compute savings vs upper: 30% cheaper
  Wasted votes saved      : -4
  Refiner: 11 triggered, 3 correct (27.3%)

  Strategy breakdown:
    majority              :   39 questions    53.8% accuracy
    coin_flip             :    7 questions    42.9% accuracy
    refiner_tiebreak      :    4 questions    50.0% accuracy

Saved -> /kaggle/working/aqua_eval/angle1_compute_efficiency.json


In [16]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

def bucket(scores):
    return {
        "all_wrong  (0%)":  sum(1 for s in scores if s == 0.0),
        "low       (1-39%)":sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

g_corr = [r["vote_consistency"] for r in all_results  if r["correct"]]
b_corr = [r["vote_consistency"] for r in base_results if r["correct"]]

# Letter distribution across votes (shows if model has preference bias)
all_letters_guided   = []
all_letters_baseline = []
for r in all_results:
    all_letters_guided.extend(r["vote_counts"].keys())
for r in base_results:
    all_letters_baseline.extend(r["vote_counts"].keys())
g_letter_dist = Counter(all_letters_guided)
b_letter_dist = Counter(all_letters_baseline)

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (AQUA-RAT)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct avg)")
print(f"    Lift     : {lift:.2f}x")
print(f"\n  Per-question: Guided wins {guided_wins}, Baseline wins {baseline_wins}, Tied {tied}")
print(f"\n  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gv, bv = g_dist[bkt], b_dist[bkt]
    sign = "+" if gv-bv >= 0 else ""
    print(f"  {bkt:<22} | {gv:>8} | {bv:>8} | {sign+str(gv-bv):>6}")

if g_corr:
    print(f"\n  Correct-question consistency: Guided={np.mean(g_corr)*100:.1f}%  Baseline={np.mean(b_corr)*100:.1f}%")

# Letter bias check -- important for multiple choice
total_guided   = sum(g_letter_dist.values())
total_baseline = sum(b_letter_dist.values())
print(f"\n  Letter vote distribution (check for A-E bias):")
print(f"  {'Letter':<8} | {'Guided':>10} | {'Baseline':>10}")
print(f"  {'-'*8}-+-{'-'*10}-+-{'-'*10}")
for letter in "ABCDE":
    gv = g_letter_dist.get(letter, 0)
    bv = b_letter_dist.get(letter, 0)
    gp = gv/total_guided*100 if total_guided else 0
    bp = bv/total_baseline*100 if total_baseline else 0
    print(f"  {letter:<8} | {gp:>9.1f}% | {bp:>9.1f}%")

angle2 = {
    "dataset": "AQUA-RAT", "n_questions": len(all_results),
    "guided_mean_consistency": round(g_mean,4), "baseline_mean_consistency": round(b_mean,4),
    "consistency_lift": round(lift,4), "guided_wins": guided_wins,
    "baseline_wins": baseline_wins, "tied": tied,
    "guided_distribution": g_dist, "baseline_distribution": b_dist,
    "guided_correct_q_consistency": round(np.mean(g_corr),4) if g_corr else 0,
    "baseline_correct_q_consistency": round(np.mean(b_corr),4) if b_corr else 0,
    "guided_letter_dist": dict(g_letter_dist),
    "baseline_letter_dist": dict(b_letter_dist),
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")


ANGLE 2 -- VOTE CONSISTENCY  (AQUA-RAT)

  Mean correct-vote ratio (out of 5 votes):
    Guided   : 40.9%  (2.04 votes correct avg)
    Baseline : 40.8%  (2.04 votes correct avg)
    Lift     : 1.00x

  Per-question: Guided wins 22, Baseline wins 16, Tied 12

  Bucket                 |   Guided | Baseline |   Diff
  -----------------------+----------+----------+-------
  all_wrong  (0%)        |       11 |       11 |     +0
  low       (1-39%)      |       13 |       15 |     -2
  medium  (40-79%)       |       19 |       11 |     +8
  high   (80-100%)       |        7 |       13 |     -6

  Correct-question consistency: Guided=64.9%  Baseline=76.5%

  Letter vote distribution (check for A-E bias):
  Letter   |     Guided |   Baseline
  ---------+------------+-----------
  A        |      21.3% |      17.4%
  B        |      16.4% |      14.8%
  C        |      23.8% |      25.2%
  D        |      16.4% |      11.3%
  E        |      22.1% |      31.3%

Saved -> /kaggle/working/aqua_ev

In [17]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# Note for multiple choice: expected accuracy at maximum confidence
# (all 5 votes agree) should approach the model's best accuracy.
# Random chance is 20% so even low confidence should beat 20%.
# False confidence is especially damaging here -- all 5 votes
# agree on a wrong letter, giving no signal to distrust the answer.

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40, 0.25),
    ]
    n_total    = len(results)
    ece        = 0.0
    calib_out  = []
    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")
    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({"bucket":name,"count":n,"accuracy":round(acc,4),
                           "expected":mid,"gap":round(gap,4)})

    hc = [r for r in results if r["confidence"] >= 0.80]
    hc_acc = sum(r["correct"] for r in hc) / max(1, len(hc)) * 100
    print(f"  {'ECE':<26}   {ece:.4f}")
    print(f"  High-conf: {len(hc)} questions  |  Accuracy: {hc_acc:.1f}%  |  Confidently WRONG: {false_conf}")
    return ece, calib_out, false_conf


print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (AQUA-RAT)")
print("=" * 65)
print("Note: random chance = 20% -- any bucket above 20% is non-trivial.")
print("False confidence here means all 5 votes picked the same wrong letter.")

g_ece, g_calib, g_false = calibration_report(all_results,  "GUIDED pipeline")
b_ece, b_calib, b_false = calibration_report(base_results, "BASELINE (no plan)")

improve = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE Summary:")
print(f"    Guided   ECE : {g_ece:.4f}")
print(f"    Baseline ECE : {b_ece:.4f}")
print(f"    Improvement  : {improve:.1f}% better calibrated")
print(f"\n  False Confidence: Guided={g_false}  Baseline={b_false}  Reduction={b_false-g_false}")

angle3 = {
    "dataset": "AQUA-RAT", "n_questions": len(all_results),
    "random_chance_pct": 20.0,
    "guided_ece": round(g_ece,4), "baseline_ece": round(b_ece,4),
    "ece_improvement_pct": round(improve,2),
    "guided_false_confidence": g_false, "baseline_false_confidence": b_false,
    "false_conf_reduction": b_false-g_false,
    "guided_calibration": g_calib, "baseline_calibration": b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")


ANGLE 3 -- CONFIDENCE CALIBRATION  (AQUA-RAT)
Note: random chance = 20% -- any bucket above 20% is non-trivial.
False confidence here means all 5 votes picked the same wrong letter.

  [GUIDED pipeline]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |    13 |     53.8% |       90% |  0.362 | Poor
  High       (0.60-0.80)     |    16 |     62.5% |       70% |  0.075 | Good
  Medium     (0.40-0.60)     |    17 |     52.9% |       50% |  0.029 | Good
  Low        (<0.40)         |     4 |      0.0% |       25% |  0.250 | Poor
  ECE                          0.1480
  High-conf: 13 questions  |  Accuracy: 53.8%  |  Confidently WRONG: 6

  [BASELINE (no plan)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |    21 |     61.9% |

In [18]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n = a1["n_questions"]

print("=" * 68)
print("  AQUA-RAT EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset: AQUA-RAT  |  N={n}  |  Seed={CONFIG['random_seed']}")
print(f"  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print(f"  Random chance baseline: 20.0%  (5 options)")
print()

rows = [
    ["Metric",                   "Baseline",      "Guided",         "Change"],
    ["Overall Accuracy",
     str(a1['baseline_accuracy']) + "%",
     str(a1['guided_accuracy']) + "%",
     "+" + str(round(a1['guided_accuracy']-a1['baseline_accuracy'],1)) + " pts"],
    ["Above Random Chance (20%)",
     "+" + str(a1['baseline_above_chance']) + " pts",
     "+" + str(a1['guided_above_chance']) + " pts",
     ""],
    ["Compute Cost",
     str(a1['baseline_compute_B']) + "B param-passes",
     str(a1['guided_compute_B']) + "B param-passes",
     str(a1['compute_savings_pct']) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1['baseline_wasted_votes']),
     str(a1['guided_wasted_votes']),
     str(a1['wasted_votes_saved']) + " fewer"],
    ["Vote Consistency",
     str(round(a2['baseline_mean_consistency']*100,1)) + "%",
     str(round(a2['guided_mean_consistency']*100,1)) + "%",
     str(round(a2['consistency_lift'],2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2['baseline_distribution']['high   (80-100%)']),
     str(a2['guided_distribution']['high   (80-100%)']),
     ""],
    ["Guided Wins Per-Question",
     "--",
     str(a2['guided_wins']) + " / " + str(n),
     ""],
    ["ECE (lower = better)",
     str(a3['baseline_ece']),
     str(a3['guided_ece']),
     str(a3['ece_improvement_pct']) + "% better"],
    ["False Confidence Count",
     str(a3['baseline_false_confidence']),
     str(a3['guided_false_confidence']),
     str(a3['false_conf_reduction']) + " fewer"],
]

col_w = [28, 20, 20, 28]
sep   = "-+-".join("-" * w for w in col_w)
for i, row in enumerate(rows):
    line = " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row))
    print("  " + line)
    if i == 0:
        print("  " + sep)

if a1.get("refiner_triggered", 0) > 0:
    rt = a1["refiner_triggered"]
    rc = a1.get("refiner_correct", 0)
    print(f"\n  Refiner: triggered {rt} times, resolved {rc} correctly ({rc/rt*100:.1f}%)")

full = {
    "dataset": "AQUA-RAT", "seed": CONFIG["random_seed"],
    "n_questions": n, "random_chance_pct": 20.0,
    "angle1": a1, "angle2": a2, "angle3": a3,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(full, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")
print("Commit this notebook to preserve outputs.")


  AQUA-RAT EVALUATION -- PAPER SUMMARY TABLE
  Dataset: AQUA-RAT  |  N=50  |  Seed=42
  Models : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver
  Random chance baseline: 20.0%  (5 options)

  Metric                       | Baseline             | Guided               | Change                      
  -----------------------------+----------------------+----------------------+-----------------------------
  Overall Accuracy             | 40.0%                | 52.0%                | +12.0 pts                   
  Above Random Chance (20%)    | +20.0 pts            | +32.0 pts            |                             
  Compute Cost                 | 7.5B param-passes    | 10.5B param-passes   | 30.0% cheaper than ceiling  
  Wasted Votes                 | 78                   | 82                   | -4 fewer                    
  Vote Consistency             | 40.8%                | 40.9%                | 1.0x lift                   
  High-Agreement Questions     | 13                   | 7   